**参考答案版**。请先独立完成 `lessons/` 里的练习,再来对照。

# Day 12 · 评分者建模：Dawid-Skene 与分歧的来源

**心理大模型评测与 Agent** · 评测方法学 · 建议用时 3 到 4 小时

**今天的目标**

- 写出 Dawid-Skene 模型，解释它如何在没有金标准时估计评分者准确率
- 比较多数投票与 Dawid-Skene 的结果
- 区分评分者能力不足和题目本身有分歧
- 知道多面 Rasch 模型能做什么

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [ ]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 模拟五位咨询师的风险等级标注

类别：0 = 低风险，1 = 中风险，2 = 高风险。评分者 4 和 5 不太准；另有 10% 的消息本身模糊（所有人都只有一半概率判对）。

In [ ]:
from course.dawid_skene import simulate_raters, dawid_skene, rater_accuracy, item_entropy
acc_true = [0.9, 0.85, 0.8, 0.6, 0.55]
L, true, amb = simulate_raters(500, acc_true, prior=[0.6, 0.3, 0.1], ambiguous_share=0.10, seed=12)
pd.DataFrame(L[:8], columns=[f"r{i+1}" for i in range(5)]).assign(true=true[:8], ambiguous=amb[:8])

## 2. 多数投票

In [ ]:
def majority(L, K=3):
    return np.array([np.bincount(row, minlength=K).argmax() for row in L])
mv = majority(L)
print("多数投票准确率:", round((mv == true).mean(), 3))

## 3. Dawid-Skene

In [ ]:
res = dawid_skene(L, 3)
ds = res["posterior"].argmax(1)
acc_hat = rater_accuracy(res["confusion"], res["prior"])
print("Dawid-Skene 准确率:", round((ds == true).mean(), 3))
pd.DataFrame({"rater": [f"r{i+1}" for i in range(5)], "true_accuracy": acc_true, "estimated": acc_hat}).round(3)

**练习 12.1**:Dawid-Skene 比多数投票的准确率高多少？

In [ ]:
ans_12_1 = float((ds == true).mean() - (mv == true).mean())
check("练习 12.1", ans_12_1, float((ds == true).mean() - (mv == true).mean()), tol=1e-9)

**练习 12.2**:Dawid-Skene 估计的最不准确的评分者是谁？填下标（0 开始）。

In [ ]:
ans_12_2 = int(np.argmin(acc_hat))
check("练习 12.2", ans_12_2, int(np.argmin(acc_hat)))

## 4. 用熵找出模糊的题

取熵最高的 50 道题，看其中有多少是真正模糊的题（`amb == True`）。

In [ ]:
H = item_entropy(res["posterior"])
top50 = np.argsort(-H)[:50]
print("熵最高 50 题中真正模糊的比例:", round(amb[top50].mean(), 3), " 全部题的模糊比例:", round(amb.mean(), 3))

**练习 12.3**：熵最高的 50 题中，模糊题的比例是总体比例的几倍？

In [ ]:
ans_12_3 = float(amb[top50].mean() / amb.mean())
check("练习 12.3", ans_12_3, float(amb[top50].mean() / amb.mean()), tol=0.01)

## 5. 只有三位评分者时

**练习 12.4**：只用前三位评分者（`L[:, :3]`）重跑。Dawid-Skene 还比多数投票好吗？填 `True`（好）或 `False`。然后写一句话解释。

In [ ]:
res3 = dawid_skene(L[:, :3], 3)
better = bool((res3["posterior"].argmax(1) == true).mean() > (majority(L[:, :3]) == true).mean())
ans_12_4 = better
check("练习 12.4", ans_12_4, better)

*解释：*

## 口述题（录音，每题 90 秒）

1. 你如何区分评分者能力不足和题目本身有分歧？
   <details><summary>参考要点</summary>看分歧的结构：某位评分者在大多数题上偏离共识，是评分者问题；某些题上所有人都分散，是题目问题。用 Dawid-Skene 估计评分者混淆矩阵，用题目后验的熵找出模糊题。前者可以过滤，后者不能。</details>
2. 没有金标准时，怎么知道众包标注的质量？
   <details><summary>参考要点</summary>用 Dawid-Skene 这类潜变量模型从重复标注中同时估计真标签和评分者误差率；用少量专家标注做校准和抽查。</details>

## 研究延伸

McBain 等人（2025）的研究发现中间风险等级最难判断。可以用 Dawid-Skene 或多面 Rasch 模型分析咨询师对中间风险消息的分歧：是评分者差异，还是这些消息本身模糊？这对设计风险筛查评测集很有用。

In [ ]:
summary()